# LiteALPR Full Pipeline: Train, Eval, Infer, Export

This notebook provides a complete walkthrough from setting up the environment to training, evaluating, running inference and exporting ONNX grouped by Detection (YOLOv8n-Efficient) and Recognition (SVTR26-Tiny) models.

## Setup Environment

Run these cells first to set up the repository, install dependencies, and download pre-trained weights.

In [1]:
!git clone https://github.com/vn-anhnth/LiteALPR.git

Cloning into 'LiteALPR'...
remote: Enumerating objects: 581, done.
remote: Counting objects: 100% (462/462), done.
remote: Compressing objects: 100% (284/284), done.
remote: Total 581 (delta 197), reused 371 (delta 135), pack-reused 119 (from 1)
Receiving objects: 100% (581/581), 26.50 MiB | 39.10 MiB/s, done.
Resolving deltas: 100% (197/197), done.


In [2]:
cd LiteALPR

/kaggle/working/LiteALPR


In [3]:
!pip install -r requirements-gpu.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of tifffile to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of tifffile to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 15.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.1/301.1 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 79.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 28.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.8/963.8 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
!wget -O pretrained_models/det/yolov8n_efficient/best.pt https://huggingface.co/anhone3/LiteALPR/resolve/main/yolov8n_efficient/best.pt
!wget -O pretrained_models/rec/svtr26_tiny/best.pth https://huggingface.co/anhone3/LiteALPR/resolve/main/svtr26_tiny/best.pth

--2026-09-13 03:07:09--  https://huggingface.co/anhone3/LiteALPR/resolve/main/yolov8n_efficient/best.pt
Resolving huggingface.co (huggingface.co)... 3.171.171.6, 3.171.171.128, 3.171.171.65, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.6|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/6a9d36a6bd37eebfbf2e42ea/62adf5c7bf1bacc398ee34e0ff3f5799dfe84dce4ed4b96eff8e8d8355f49d58?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27best.pt%3B+filename%3D%22best.pt%22%3B&X-Xet-Cas-Uid=public&user_id=public&Expires=1789272429&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNmE5ZDM2YTZiZDM3ZWViZmJmMmU0MmVhLzYyYWRmNWM3YmYxYmFjYzM5OGVlMzRlMGZmM2Y1Nzk5ZGZlODRkY2U0ZWQ0Yjk2ZWZmOGU4ZDgzNTVmNDlkNThcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomWC1YZXQtQ2FzLVVpZD1wdWJsaWMmdXNlcl9pZD1wdWJsaWMiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGltZSI6MTc4OTI3MjQyOX19fV19

In [5]:
!python tools/create_lmdb_dataset.py \
    --data_dir ./dataset/rec \
    --label_files train_labels.txt val_labels.txt test_labels.txt \
    --output_dir ./dataset/rec/lmdb_data

Creating LMDB dataset at: ./dataset/rec/lmdb_data/train
load data from ./dataset/rec/train_labels.txt: 100%|█| 5/5 [00:00<00:00, 44525.5
make dataset, save to ./dataset/rec/lmdb_data/train: 100%|█| 5/5 [00:00<00:00, 1
Created dataset with 5 samples
Creating LMDB dataset at: ./dataset/rec/lmdb_data/val
load data from ./dataset/rec/val_labels.txt: 100%|█| 2/2 [00:00<00:00, 26379.27i
make dataset, save to ./dataset/rec/lmdb_data/val: 100%|█| 2/2 [00:00<00:00, 461
Created dataset with 2 samples
Creating LMDB dataset at: ./dataset/rec/lmdb_data/test
load data from ./dataset/rec/test_labels.txt: 100%|█| 5/5 [00:00<00:00, 53362.65
make dataset, save to ./dataset/rec/lmdb_data/test: 100%|█| 4/4 [00:00<00:00, 64
Created dataset with 4 samples


## Part 1: Detection Model (YOLOv8n-Efficient)

All workflows for the License Plate Detection model.

### 1.1 Train Detection

In [6]:
!python tools/train_det.py -c configs/det/yolov8/yolov8n_efficient.yml

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Transferred 523/523 items from pretrained weights
New https://pypi.org/project/ultralytics/8.4.150 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.99 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: task=detect, mode=train, model=/kaggle/working/LiteALPR/configs/det/yolov8/yolov8n_efficient.yml, data=/kaggle/working/LiteALPR/dataset/det/data.yaml, epochs=50, time=None, patience=100, batch=256, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=output/det/yolov8n_efficient, name=train, exist_ok=False, pretrained=pretrained_models/det/yolov8n_efficient/best.pt, optimizer=auto, verbose=True, seed=0, dete

### 1.2 Evaluate Detection

In [8]:
!python tools/eval_det.py -m output/det/yolov8n_efficient/train/weights/best.pt

[INFO] Starting evaluation.
Ultralytics 8.3.99 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8n_efficient summary (fused): 140 layers, 1,993,871 parameters, 0 gradients, 5.6 GFLOPs
val: Scanning /kaggle/working/LiteALPR/dataset/det/val/labels.cache... 4 images,
                 Class     Images  Instances      Box(P          R      mAP50  m
                   all          4          4          1          1      0.995      0.961
Speed: 0.3ms preprocess, 12.3ms inference, 0.0ms loss, 37.5ms postprocess per image
Results saved to output/det/yolov8n_efficient/val2

[INFO] ========================================
[INFO] Evaluation completed!
[INFO] Precision  : 1.0000
[INFO] Recall     : 1.0000
[INFO] mAP50      : 0.9950
[INFO] mAP50-95   : 0.9611
[INFO] Results saved to: output/det/yolov8n_efficient/eval
[INFO] ========================================


### 1.3 Infer Detection
Run detection on a directory of images and save visualizations.

In [9]:
!python tools/infer_det.py -m output/det/yolov8n_efficient/train/weights/best.pt -d dataset/det/test/images --save_log

[INFO] Initializing inference on device: CUDA:0
[INFO] Loading model: output/det/yolov8n_efficient/train/weights/best.pt
[INFO] Found 5 images in dataset/det/test/images.
[INFO] Pre-loading images into memory...
[INFO] Starting warmup (10 iterations)...
[INFO] Warmup completed. Measuring FPS...

[INFO] ========================================
[INFO] Inference Test
[INFO] Device        : CUDA:0
[INFO] Total Images  : 5
[INFO] Avg Time/Img  : 10.83 ms
[INFO] FPS           : 92.33 frames/sec
[INFO] Log saved     : output/det/yolov8n_efficient/infer/infer.log
[INFO] ========================================



### 1.4 Export ONNX

In [10]:
!python tools/export_det.py -m output/det/yolov8n_efficient/train/weights/best.pt --imgsz 416 --opset 18

[INFO] Initializing export on device: cuda:0
[INFO] Loading detection model: output/det/yolov8n_efficient/train/weights/best.pt
[INFO] Exporting to ONNX (imgsz=416, opset=18, FP32)...
Ultralytics 8.3.99 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon 2.00GHz)
YOLOv8n_efficient summary (fused): 140 layers, 1,993,871 parameters, 0 gradients, 5.6 GFLOPs

PyTorch: starting from 'output/det/yolov8n_efficient/train/weights/best.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 5, 3549) (4.1 MB)

ONNX: starting export with onnx 1.22.0 opset 18...
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 6.9s, saved as 'output/det/yolov8n_efficient/train/weights/best.onnx' (7.7 MB)

Export complete (8.6s)
Results saved to /kaggle/working/LiteALPR/output/det/yolov8n_efficie

## Part 2: Recognition Model (SVTR26-Tiny)

All workflows for the License Plate Text Recognition model.

### 2.1 Train Recognition

In [11]:
!torchrun --nproc_per_node=1 tools/train_rec.py \
    -c configs/rec/svtr26/svtr26_tiny.yml

[2026/09/13 03:10:18] rec INFO: ----------- Config -----------
[2026/09/13 03:10:18] rec INFO: Architecture : 
[2026/09/13 03:10:18] rec INFO:     Decoder : 
[2026/09/13 03:10:18] rec INFO:         bottleneck_channels : 128
[2026/09/13 03:10:18] rec INFO:         name : EfficientRCTCDecoder
[2026/09/13 03:10:18] rec INFO:     Encoder : 
[2026/09/13 03:10:18] rec INFO:         depths : [3, 6, 3]
[2026/09/13 03:10:18] rec INFO:         dims : [64, 128, 256]
[2026/09/13 03:10:18] rec INFO:         feat2d : True
[2026/09/13 03:10:18] rec INFO:         last_stage : False
[2026/09/13 03:10:18] rec INFO:         mixer : [['Conv', 'Conv', 'Conv'], ['Conv', 'Conv', 'Conv', 'FGlobal', 'Global', 'Global'], ['Global', 'Global', 'Global']]
[2026/09/13 03:10:18] rec INFO:         name : SVTRv2LNConvTwo33
[2026/09/13 03:10:18] rec INFO:         num_heads : [2, 4, 8]
[2026/09/13 03:10:18] rec INFO:         sub_k : [[1, 1], [2, 1], [-1, -1]]
[2026/09/13 03:10:18] rec INFO:         use_pos_embed : False

### 2.2 Evaluate Recognition

In [12]:
!python tools/eval_rec.py -c configs/rec/svtr26/svtr26_tiny.yml -m output/rec/svtr26_tiny/train/best.pth

[2026/09/13 03:11:43] rec INFO: ----------- Config -----------
[2026/09/13 03:11:43] rec INFO: Architecture : 
[2026/09/13 03:11:43] rec INFO:     Decoder : 
[2026/09/13 03:11:43] rec INFO:         bottleneck_channels : 128
[2026/09/13 03:11:43] rec INFO:         name : EfficientRCTCDecoder
[2026/09/13 03:11:43] rec INFO:     Encoder : 
[2026/09/13 03:11:43] rec INFO:         depths : [3, 6, 3]
[2026/09/13 03:11:43] rec INFO:         dims : [64, 128, 256]
[2026/09/13 03:11:43] rec INFO:         feat2d : True
[2026/09/13 03:11:43] rec INFO:         last_stage : False
[2026/09/13 03:11:43] rec INFO:         mixer : [['Conv', 'Conv', 'Conv'], ['Conv', 'Conv', 'Conv', 'FGlobal', 'Global', 'Global'], ['Global', 'Global', 'Global']]
[2026/09/13 03:11:43] rec INFO:         name : SVTRv2LNConvTwo33
[2026/09/13 03:11:43] rec INFO:         num_heads : [2, 4, 8]
[2026/09/13 03:11:43] rec INFO:         sub_k : [[1, 1], [2, 1], [-1, -1]]
[2026/09/13 03:11:43] rec INFO:         use_pos_embed : False

### 2.3 Infer Recognition
Run text recognition directly on cropped plate images.

In [13]:
!python tools/infer_rec.py -m output/rec/svtr26_tiny/train/best.pth -d dataset/rec/test --save_log

[INFO] Initializing inference on device: CUDA
[INFO] Loading model checkpoint: output/rec/svtr26_tiny/train/best.pth
[INFO] Found 5 images in dataset/rec/test.
[INFO] Pre-loading images into tensors to isolate pure model inference time...
[INFO] Starting warmup (10 iterations)...
[INFO] Warmup completed. Measuring FPS...

[INFO] ========================================
[INFO] Inference Test
[INFO] Device        : CUDA
[INFO] Total Images  : 5
[INFO] Avg Time/Img  : 7.27 ms
[INFO] FPS           : 137.48 frames/sec
[INFO] Log saved     : output/rec/train/infer/infer.log
[INFO] ========================================



### 2.4 Export ONNX

In [14]:
!python tools/export_rec.py -m output/rec/svtr26_tiny/train/best.pth --save_path output/rec/svtr26_tiny/train/best.onnx --opset 18 --dynamic

[INFO] Initializing export on device: CUDA
[INFO] Loading model checkpoint: output/rec/svtr26_tiny/train/best.pth
[INFO] Exporting to ONNX (input=32x128, opset=18, FP32, dynamic=True)...
[torch.onnx] Obtain model graph for `BaseRecognizer([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BaseRecognizer([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[INFO] Checking ONNX model...
[INFO] ONNX export successful!
[INFO] ONNX opset: 18
[INFO] ONNX model: output/rec/svtr26_tiny/train/best.onnx
